# TimesFM 2.5 Torch Finetuning on Financial Data

This notebook replicates the v1 financial finetuning example using the new `src/timesfm` finetuning APIs.

It trains on AAPL close prices from Yahoo Finance and uses the new classes under `timesfm.finetuning`.


## Environment

Required packages:
- `timesfm[torch,finetune]`
- `yfinance`
- `pandas`
- `matplotlib`


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yfinance as yf

import timesfm
from timesfm.finetuning import (
    FinetuningConfig,
    TimeSeriesWindowDataset,
    TimesFMFinetuner,
    TimesFMTorchTrainAdapter,
    WindowingConfig,
    create_collate_fn,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


In [2]:
def download_aapl_data(start: str = "2010-01-01", end: str = "2019-01-01") -> np.ndarray:
    df = yf.download("AAPL", start=start, end=end, auto_adjust=True)
    if df.empty or "Close" not in df.columns:
        raise RuntimeError("Failed to download AAPL close prices from yfinance.")
    series = df["Close"].astype(float).to_numpy()
    if len(series) == 0:
        raise RuntimeError("Downloaded AAPL series is empty.")
    return series

series = download_aapl_data()
print(f"Downloaded points: {len(series)}")
print(f"First 3 values: {series[:3]}")


[*********************100%***********************]  1 of 1 completed

Downloaded points: 2264
First 3 values: [[6.41238165]
 [6.42346907]
 [6.32129526]]


In [3]:
def build_datasets(
    values: np.ndarray,
    context_len: int = 256,
    horizon_len: int = 64,
    stride: int = 1,
    train_split: float = 0.8,
):
    split_idx = int(len(values) * train_split)

    # Keep enough overlap for validation windows.
    train_values = values[:split_idx]
    val_start = max(0, split_idx - context_len - horizon_len)
    val_values = values[val_start:]

    window_cfg = WindowingConfig(
        context_len=context_len,
        horizon_len=horizon_len,
        stride=stride,
    )

    train_ds = TimeSeriesWindowDataset([train_values], window_cfg)
    val_ds = TimeSeriesWindowDataset([val_values], window_cfg)
    return train_ds, val_ds, window_cfg

train_dataset, val_dataset, window_cfg = build_datasets(series)
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(window_cfg)


Train samples: 1492
Val samples:   454
WindowingConfig(context_len=256, horizon_len=64, stride=1, min_context=1, drop_last=True)


In [4]:
# Default: load from Hugging Face model id.
# Set LOCAL_MODEL_DIR to a local checkpoint directory with model.safetensors to avoid download.
MODEL_ID = "google/timesfm-2.5-200m-pytorch"
LOCAL_MODEL_DIR = None  # e.g. "/path/to/local/timesfm-2p5"

model_source = LOCAL_MODEL_DIR if LOCAL_MODEL_DIR else MODEL_ID
wrapper = timesfm.TimesFM_2p5_200M_torch.from_pretrained(model_source)
adapter = TimesFMTorchTrainAdapter(wrapper.model)
print(f"Loaded model from: {model_source}")
print(f"Patch len: {adapter.patch_len}, Max train horizon (phase-1): {adapter.max_horizon}")



config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

Downloaded.
Loaded model from: google/timesfm-2.5-200m-pytorch
Patch len: 32, Max train horizon (phase-1): 128


In [6]:
output_dir = Path("outputs/notebook_finetune")
output_dir.mkdir(parents=True, exist_ok=True)

finetune_cfg = FinetuningConfig(
    batch_size=16,
    num_epochs=2,
    learning_rate=1e-4,
    weight_decay=1e-2,
    mixed_precision=(DEVICE == "cuda"),
    log_every_n_steps=25,
    output_dir=str(output_dir),
    device=DEVICE,
)

trainer = TimesFMFinetuner(adapter, finetune_cfg)



In [9]:
series

array([[ 6.41238165],
       [ 6.42346907],
       [ 6.32129526],
       ...,
       [37.08337784],
       [37.1023674 ],
       [37.46098709]], shape=(2264, 1))

In [ ]:
result = trainer.finetune(train_dataset, val_dataset)
print(result)

In [ ]:
def plot_prediction_example(
    model_adapter: TimesFMTorchTrainAdapter,
    dataset: TimeSeriesWindowDataset,
    horizon_len: int,
    figure_path: Path,
) -> None:
    model_adapter.eval()
    collate = create_collate_fn(model_adapter.patch_len)
    batch = collate([dataset[0]])

    context = batch["context"].to(next(model_adapter.parameters()).device)
    context_mask = batch["context_mask"].to(next(model_adapter.parameters()).device)
    target = batch["target"][0].cpu().numpy()

    with torch.no_grad():
        out = model_adapter.forward_train(context=context, context_mask=context_mask, horizon=horizon_len)

    context_np = context[0].detach().cpu().numpy()
    forecast_np = out.point_forecast[0].detach().cpu().numpy()

    # Remove front padding for visualization.
    valid_start = np.where(~batch["context_mask"][0].numpy())[0][0]
    context_np = context_np[valid_start:]

    plt.figure(figsize=(12, 5))
    x0 = np.arange(len(context_np))
    x1 = np.arange(len(context_np), len(context_np) + len(target))
    plt.plot(x0, context_np, label="Context", linewidth=2)
    plt.plot(x1, target, label="Ground Truth", linewidth=2, linestyle="--")
    plt.plot(x1, forecast_np, label="Forecast", linewidth=2)
    plt.title("TimesFM 2.5 Finetuning Example (AAPL)")
    plt.xlabel("Time step")
    plt.ylabel("Close price")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_path)
    plt.show()

fig_path = output_dir / "prediction_plot.png"
plot_prediction_example(adapter, val_dataset, window_cfg.horizon_len, fig_path)
print(f"Saved plot to: {fig_path.resolve()}")


In [ ]:
# Optional: verify base inference API still works with compile + forecast.
infer_model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(model_source)
infer_model.compile(
    timesfm.ForecastConfig(
        max_context=256,
        max_horizon=64,
        normalize_inputs=True,
    )
)
point, quantile = infer_model.forecast(horizon=8, inputs=[series[-200:]])
print(point.shape, quantile.shape)
